In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

# ==========================================
# 1. 超参数配置
# ==========================================
learning_rate = 1e-4
# TF1.x 的 keep_prob 是保留概率 (0.7)，TF2.x 的 Dropout 参数是丢弃率 (0.3)
dropout_rate = 0.3
batch_size = 100
epochs = 3  # 原代码的 2000 是 steps，这里换算为合理的 epochs

# ==========================================
# 2. 数据加载与预处理
# ==========================================
print("Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 增加通道维度使其变为 [batch, 28, 28, 1]，并归一化到 0-1
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

# ==========================================
# 3. 构建模型架构 (Sequential API)
# ==========================================
# 现代 TF 不再需要手动定义 W 和 b 张量，底层机制由 Keras Layer 自动管理
model = models.Sequential(
    [
        # 卷积层 1 (patch 7x7, in 1, out 32, SAME padding)
        layers.Conv2D(
            filters=32,
            kernel_size=(7, 7),
            padding="same",
            activation="relu",
            input_shape=(28, 28, 1),
        ),
        # 池化层 1 (2x2, stride 2, SAME padding)
        layers.MaxPooling2D(pool_size=(2, 2), strides=2, padding="same"),
        # 卷积层 2 (patch 5x5, in 32, out 64, SAME padding)
        layers.Conv2D(
            filters=64, kernel_size=(5, 5), padding="same", activation="relu"
        ),
        # 池化层 2 (2x2, stride 2, SAME padding)
        layers.MaxPooling2D(pool_size=(2, 2), strides=2, padding="same"),
        # 展平层：自动推断维度为 7*7*64，无需手动计算并 reshape
        layers.Flatten(),
        # 全连接层 1
        layers.Dense(units=1024, activation="relu"),
        layers.Dropout(rate=dropout_rate),
        # 全连接层 2 (输出层)
        layers.Dense(units=10, activation="softmax"),
    ]
)

# ==========================================
# 4. 编译与训练模型
# ==========================================
# 使用 sparse_categorical_crossentropy 因为我们的标签是 0-9 的整数，而非 One-hot 编码
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# 打印模型结构概览
model.summary()

# model.fit 自动处理 batch 分发和循环，取代了繁杂的 sess.run()
print("\nStarting training...")
history = model.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test, y_test),  # 每一轮结束自动在测试集上评估
)

Loading MNIST dataset...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 9s 1us/step


c:\Users\30663\anaconda3\envs\DeepLearning\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     3,212,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,275,402 (12.49 MB)

 Trainable params: 3,275,402 (12.49 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/3
600/600 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9015 - loss: 0.3677 - val_accuracy: 0.9681 - val_loss: 0.1038
Epoch 2/3
600/600 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9715 - loss: 0.0964 - val_accuracy: 0.9808 - val_loss: 0.0599
Epoch 3/3
600/600 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.9799 - loss: 0.0662 - val_accuracy: 0.9845 - val_loss: 0.0486
